In [1]:
import sys, os
sys.path.insert(0, '/home/zaid/Documents/My_Deep_Learning/GenAI/projects/mini_llm/')

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

In [3]:
import re
import math
import pickle
from collections import defaultdict
from typing import List, Dict

###### 

In [4]:
class Tokenizer:

    def __init__(self, num_merges, max_seq_len):

        self.dataset = None

        self.num_merges = num_merges
        self.max_seq_len = max_seq_len

        # =====================================================
        # Core Storage
        # =====================================================

        self.vocab = {}
        self.bytes_to_ids = {}

        # =====================================================
        # Special Tokens
        # =====================================================

        self.special_tokens = {
            "[BOS]": None,
            "[EOS]": None,
            "[PAD]": None,
            "[UNK]": None
        }

        self.special_token_ids = {}

        self.is_trained = False

    # =========================================================
    # Dataset Upload
    # =========================================================

    def upload_dataset(self, path: str):
        with open(path, "r") as f:
            self.dataset = f.read()
        
    
    # =========================================================
    # Utilities
    # =========================================================

    @staticmethod
    def get_byte_representation(text: str) -> List[int]:
        return list(text.encode("utf-8"))

    @staticmethod
    def decode_byte_representation(token: List[int]) -> str:
        return bytes(token).decode(
            "utf-8",
            errors="replace"
        )

    @staticmethod
    def pretokenize(text: str) -> List[str]:

        return re.findall(
            r"\s+|\w+|[^\w\s]",
            text
        )

    # =========================================================
    # Pair Frequency
    # =========================================================

    def get_pair_frequencies(self, tokens):

        pair_freq = defaultdict(int)

        for word in tokens:

            for i in range(len(word) - 1):

                pair = (
                    word[i],
                    word[i + 1]
                )

                pair_freq[pair] += 1

        return pair_freq

    # =========================================================
    # Merge Pair
    # =========================================================

    def merge_pair(self, tokens, pair, new_id):

        merged_tokens = []

        for word in tokens:

            i = 0
            merged_word = []

            while i < len(word):

                if (
                    i < len(word) - 1
                    and word[i] == pair[0]
                    and word[i + 1] == pair[1]
                ):

                    merged_word.append(new_id)
                    i += 2

                else:
                    merged_word.append(word[i])
                    i += 1

            merged_tokens.append(merged_word)

        return merged_tokens

    # =========================================================
    # Training
    # =========================================================

    def train(self, save_path):

        if not self.dataset:
            raise ValueError("Dataset not uploaded. Using upload_dataset upload the dataset path.")

        # =====================================================
        # Pretokenize
        # =====================================================

        tokens = self.pretokenize(self.dataset)

        tokens = [
            self.get_byte_representation(token)
            for token in tokens
        ]

        # =====================================================
        # Initialize Base Vocabulary
        # =====================================================

        self.vocab = {}

        for idx in range(256):
            self.vocab[idx] = [idx]

        next_token_id = 256

        # =====================================================
        # Train BPE
        # =====================================================

        for i in range(self.num_merges):

            pair_freq = self.get_pair_frequencies(tokens)

            if not pair_freq:
                break

            best_pair = max(
                pair_freq,
                key=pair_freq.get
            )

            # Build merged token bytes
            self.vocab[next_token_id] = (
                self.vocab[best_pair[0]]
                + self.vocab[best_pair[1]]
            )

            # Apply merge
            tokens = self.merge_pair(
                tokens,
                best_pair,
                next_token_id
            )

            print(
                f"Merge {i+1}: "
                f"{best_pair} -> {next_token_id}"
            )

            next_token_id += 1

        # =====================================================
        # Build Reverse Lookup
        # =====================================================

        self.bytes_to_ids = {
            tuple(value): key
            for key, value in self.vocab.items()
        }

        # =====================================================
        # Add Special Tokens
        # =====================================================

        current_vocab_size = len(self.vocab)

        self.special_tokens = {
            "[BOS]": current_vocab_size,
            "[EOS]": current_vocab_size + 1,
            "[PAD]": current_vocab_size + 2,
            "[UNK]": current_vocab_size + 3
        }

        self.special_token_ids = {
            v: k
            for k, v in self.special_tokens.items()
        }

        self.vocab.update(self.special_token_ids)
        self.bytes_to_ids.update(self.special_tokens)

        self.is_trained = True

        print(
            f"\nTraining Complete "
            f"| Vocab Size: {len(self.vocab)}"
        )

        self.save(save_path)
        print(f"Model saved at path: {save_path}")

    # =========================================================
    # Encode Single Word
    # =========================================================

    def encode_word(self, word: str):

        word_bytes = self.get_byte_representation(word)

        tokens = []

        start = 0

        while start < len(word_bytes):

            end = len(word_bytes)

            found = None

            while end > start:

                piece = tuple(
                    word_bytes[start:end]
                )

                if piece in self.bytes_to_ids:

                    found = piece
                    break

                end -= 1

            # Unknown fallback
            if found is None:

                tokens.append(
                    self.special_tokens["[UNK]"]
                )

                break

            tokens.append(
                self.bytes_to_ids[found]
            )

            start = end

        return tokens

    # =========================================================
    # Encode Sequence
    # =========================================================

    def encode(self, sequence: str, max_seq_len: int = None):

        if not self.is_trained:
            raise ValueError("Tokenizer must be trained first.")

        if max_seq_len is None:
            max_seq_len = self.max_seq_len

        tokens = self.pretokenize(sequence)

        encoded_sequence = [
            self.special_tokens["[BOS]"]
        ]

        for token in tokens:

            encoded = self.encode_word(token)

            encoded_sequence.extend(encoded)

        encoded_sequence.append(
            self.special_tokens["[EOS]"]
        )

        # =====================================================
        # Attention Mask
        # =====================================================

        attention_mask = [1] * len(encoded_sequence)

        # =====================================================
        # Padding
        # =====================================================

        if len(encoded_sequence) < max_seq_len:

            pad_count = (
                max_seq_len - len(encoded_sequence)
            )

            encoded_sequence.extend(
                [self.special_tokens["[PAD]"]]
                * pad_count
            )

            attention_mask.extend(
                [0] * pad_count
            )

        # =====================================================
        # Truncation
        # =====================================================

        elif len(encoded_sequence) > max_seq_len:

            encoded_sequence = encoded_sequence[
                :max_seq_len - 1
            ]

            encoded_sequence.append(
                self.special_tokens["[EOS]"]
            )

            attention_mask = attention_mask[
                :max_seq_len
            ]

        return {
            "input_ids": encoded_sequence,
            "attention_mask": attention_mask
        }

    # =========================================================
    # Batch Encode
    # =========================================================

    def encode_batch(self, batch: List[str]):

        return [
            self.encode(sequence)
            for sequence in batch
        ]

    # =========================================================
    # Decode
    # =========================================================

    def decode(self, sequence: List[int]):

        decoded = ""

        for token in sequence:

            # Skip special tokens
            if token in self.special_token_ids:
                continue

            decoded += self.decode_byte_representation(
                self.vocab[token]
            )

        return decoded

    # =========================================================
    # Batch Decode
    # =========================================================

    def decode_batch(self, batch):

        decoded_batch = []

        for encoded_sequence in batch:

            decoded_batch.append(
                self.decode(
                    encoded_sequence["input_ids"]
                )
            )

        return decoded_batch

    # =========================================================
    # Save
    # =========================================================

    def save(self, path: str):

        data = {
            "vocab": self.vocab,
            "special_tokens": self.special_tokens,
            "max_seq_len": self.max_seq_len,
            "num_merges": self.num_merges
        }

        with open(path, "wb") as f:
            pickle.dump(data, f)

        print(f"Tokenizer saved to: {path}")

    # =========================================================
    # Load
    # =========================================================

    @classmethod
    def load(cls, path: str):

        with open(path, "rb") as f:
            data = pickle.load(f)

        tokenizer = cls(
            num_merges=data["num_merges"],
            max_seq_len=data["max_seq_len"]
        )

        tokenizer.vocab = data["vocab"]
        tokenizer.special_tokens = data["special_tokens"]

        tokenizer.special_token_ids = {
            v: k
            for k, v in tokenizer.special_tokens.items()
        }

        tokenizer.bytes_to_ids = {
            tuple(value): key
            for key, value in tokenizer.vocab.items()
            if isinstance(value, list)
        }

        tokenizer.bytes_to_ids.update(
            tokenizer.special_tokens
        )

        tokenizer.is_trained = True

        print(f"Tokenizer loaded from: {path}")

        return tokenizer

    # =========================================================
    # Vocabulary Size
    # =========================================================

    @property
    def vocab_size(self):
        
        return len(self.vocab)


    # =========================================================
    # Get PAD token ID
    # =========================================================

    def get_pad_token_id(self):

        return self.special_tokens["[PAD]"]

In [5]:
class Embedding(nn.Module):

    def __init__(self, vocab_size, embedding_dim, device="cpu"):
        
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.device = device

        # =================================================
        # Embedding Matrix
        # =================================================

        self.weight = nn.Parameter(
            torch.randn(
                vocab_size,
                embedding_dim,
                device=device
            ) * 0.02
        )

    def forward(self, input_ids):

        # input_ids:
        # (batch_size, seq_len)

        return self.weight[input_ids]

In [6]:
class PositionalEncoding(nn.Module):

    def __init__(self, max_seq_len, embedding_dim):

        super().__init__()
        self.max_seq_len = max_seq_len
        self.embedding_dim = embedding_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # =================================================
        # Create PE Matrix
        # =================================================

        pe = torch.zeros(
            max_seq_len,
            embedding_dim,
            device=self.device
        )

        # Positions
        position = torch.arange(
            0,
            max_seq_len,
            dtype=torch.float
        ).unsqueeze(1).to(self.device)

        # Frequency scaling
        div_term = torch.exp(
            torch.arange(
                0,
                embedding_dim,
                2
            ).float()
            * (
                -torch.log(
                    torch.tensor(10000.0)
                ) / embedding_dim
            )
        ).to(self.device)

        # Even indices -> sin
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # Odd indices -> cos
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # Add batch dimension
        self.register_buffer('pe', pe.unsqueeze(0))
        

    def forward(self, x):

        # x shape:
        # (batch_size, seq_len, embedding_dim)

        seq_len = x.size(1)

        return x + self.pe[:, :seq_len]

In [7]:
class MultiHeadCausalSelfAttention(nn.Module):

    def __init__(self, embedding_dim, max_seq_len, num_heads, dropout_rate):

        super().__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.dropout = nn.Dropout(p=dropout_rate)

        # ============================================
        # Projection Matrices
        # ============================================

        self.W_q = nn.Parameter(
            torch.randn(
                embedding_dim,
                embedding_dim
            ) * 0.02
        )

        self.W_k = nn.Parameter(
            torch.randn(
                embedding_dim,
                embedding_dim
            ) * 0.02
        )

        self.W_v = nn.Parameter(
            torch.randn(
                embedding_dim,
                embedding_dim
            ) * 0.02
        )

        self.W_o = nn.Parameter(
            torch.randn(
                embedding_dim,
                embedding_dim
            ) * 0.02
        )

        self.W_ob = nn.Parameter(
            torch.zeros(
                embedding_dim
            )
        )

        # ============================================
        # Causal Mask
        # ============================================

        self.register_buffer(
            'mask', 
            torch.tril(
                torch.ones(max_seq_len, max_seq_len)
            )
        )

    def forward(self, x, attention_mask):

        # x shape:
        # (B,T,C)

        B, T, C = x.shape

        # ============================================
        # Create Q,K,V
        # ============================================

        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v

        # ============================================
        # Split into heads
        # ============================================

        Q = Q.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Shape:
        # (B,H,T,D)

        # ============================================
        # Attention Scores
        # ============================================

        scores = (
            Q @ K.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        # Shape:
        # (B,H,T,T)

        # ============================================
        # Apply Causal Mask
        # ============================================

        mask = self.mask[:T, :T]

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        # ============================================
        # Padding Mask
        # ============================================

        if attention_mask is not None:

            # (B,T) -> (B,1,1,T)
            padding_mask = attention_mask.unsqueeze(1).unsqueeze(2)

            scores = scores.masked_fill(
                padding_mask == 0,
                float("-inf")
            )

        # ============================================
        # Softmax
        # ============================================

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(attention_weights)

        # ============================================
        # Weighted Sum
        # ============================================

        output = attention_weights @ V

        # Shape:
        # (B,H,T,D)

        # ============================================
        # Merge Heads
        # ============================================

        output = output.transpose(1, 2)

        output = output.contiguous().view(
            B,
            T,
            C
        )

        # Shape:
        # (B,T,C)

        # ============================================
        # Output Projection
        # ============================================

        output = (output @ self.W_o) + self.W_ob

        return output

In [8]:
class FeedForward(nn.Module):

    def __init__(self, embedding_dim, hidden_dim=None, gelu_const=None, dropout_rate=0.1):

        super().__init__()

        self.embedding_dim = embedding_dim

        if hidden_dim is None:
            hidden_dim = embedding_dim * 4

        self.hidden_dim = hidden_dim

        self.dropout = nn.Dropout(p=dropout_rate)

        self.GELU_CONST = gelu_const

        # =============================================
        # Linear Layers
        # =============================================

        self.W1 = nn.Parameter(
            torch.randn(
                embedding_dim,
                hidden_dim
            ) * 0.02
        )

        self.b1 = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.W2 = nn.Parameter(
            torch.randn(
                hidden_dim,
                embedding_dim
            ) * 0.02
        )

        self.b2 = nn.Parameter(
            torch.zeros(embedding_dim)
        )

    def gelu(self, x):
        return 0.5 * x * (
            1 + torch.tanh(
                self.GELU_CONST * (
                    x + 0.044715 * torch.pow(x, 3)
                )
            )
        )

    def forward(self, x):

        # x:
        # (B,T,C)

        x = x @ self.W1 + self.b1

        x = self.gelu(x)

        x = self.dropout(x)

        x = x @ self.W2 + self.b2

        return x

In [9]:
class LayerNorm(nn.Module):

    def __init__(self, embedding_dim, eps=1e-5):

        super().__init__()

        self.embedding_dim = embedding_dim
        self.eps = eps

        # =============================================
        # Learnable Parameters
        # =============================================

        self.gamma = nn.Parameter(
            torch.ones(embedding_dim)
        )

        self.beta = nn.Parameter(
            torch.zeros(embedding_dim)
        )

    def forward(self, x):

        # x:
        # (B,T,C)

        # =============================================
        # Mean
        # =============================================

        mean = x.mean(
            dim=-1,
            keepdim=True
        )

        # =============================================
        # Variance
        # =============================================

        variance = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False
        )

        # =============================================
        # Normalize
        # =============================================

        x_hat = (
            x - mean
        ) / torch.sqrt(
            variance + self.eps
        )

        # =============================================
        # Scale + Shift
        # =============================================

        out = (
            self.gamma * x_hat
        ) + self.beta

        return out

In [10]:
class Linear(nn.Module):

    def __init__(self, in_features, out_features):

        super().__init__()

        self.in_features = in_features
        self.out_features = out_features

        # ==========================================
        # Weight Initialization
        # ==========================================
        self.weights = nn.Parameter(
            torch.randn(
                in_features,
                out_features
            ) * (1 / (in_features ** 0.5))
        )

        # ==========================================
        # Bias Initialization
        # ==========================================
        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )


    def forward(self, x):

        """
        x shape:
        (batch_size, seq_len, in_features)
        """

        out = torch.matmul(
            x, 
            self.weights
        )
        out = out + self.bias

        return out

###### 

In [11]:
class DecoderBlock(nn.Module):
    
    def __init__(self, embedding_dim, max_seq_len, num_att_heads, gelu_const, dropout_rate):

        super().__init__()
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dim,
            max_seq_len, 
            num_att_heads,
            dropout_rate
        )

        self.ffnn = FeedForward(
            embedding_dim=embedding_dim, 
            hidden_dim=embedding_dim*4, 
            gelu_const=gelu_const,
            dropout_rate=dropout_rate
        )

        self.layer_norm_1 = LayerNorm(embedding_dim)
        self.layer_norm_2 = LayerNorm(embedding_dim)

        self.dropout = nn.Dropout(p=dropout_rate)


    def forward(self, x, attention_mask):
        # Pre-LN Attention + Residual
        attention_out = self.attention(self.layer_norm_1(x), attention_mask)
        attention_out = self.dropout(attention_out)
        x = x + attention_out
    
        # Pre-LN FFNN + Residual
        ffn_out = self.ffnn(self.layer_norm_2(x))
        x = x + ffn_out
    
        return x

In [12]:
class Mini_LLM(nn.Module):

    def __init__(self, config):

        super().__init__()

        num_merges = config["num_merges"]
        self.max_seq_len = config["max_seq_len"]
        self.embedding_dim = config["embedding_dim"]
        self.num_att_heads = config["num_att_heads"]
        
        self.tokenizer = Tokenizer(num_merges, self.max_seq_len)

        if config["train_vocab"]:
            self.tokenizer.upload_dataset(config["train_dataset_path"])
            self.tokenizer.train(config["vocab_save_path"])

        else:
            vocab_path = config["vocab_path"]
            self.tokenizer = self.tokenizer.load(vocab_path)
        
        self.vocab_size = self.tokenizer.vocab_size

        self.dropout_rate = config["dropout_rate"]
        self.dropout = nn.Dropout(p=self.dropout_rate)

        self.embedding = Embedding(self.vocab_size, self.embedding_dim)
        
        self.PE = PositionalEncoding(self.max_seq_len, self.embedding_dim)

        self.gelu_const = config["gelu_const"]
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                embedding_dim=self.embedding_dim,
                max_seq_len=self.max_seq_len, 
                num_att_heads=self.num_att_heads,
                gelu_const=self.gelu_const,
                dropout_rate=self.dropout_rate
            )
            for _ in range(config["num_decoder_blocks"])
        ])

        self.output_projection = Linear(
            self.embedding_dim,
            self.vocab_size
        )

        self.final_layer_norm = LayerNorm(
            self.embedding_dim
        )

        self.loss_fn = torch.nn.CrossEntropyLoss(
            ignore_index=self.tokenizer.get_pad_token_id()
        )

        self.learning_rate = config["learning_rate"]
        self.optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.learning_rate
        )

        self.device = torch.device(
            "cuda" if torch.cuda.is_available()
            else "cpu"
        )


    def preprocess(self, data):
        encoded = self.tokenizer.encode_batch(
            data
        )
    
        input_ids = [item["input_ids"] for item in encoded]
    
        attention_masks = [item["attention_mask"] for item in encoded]
    
        tokens = torch.tensor(
            input_ids,
            dtype=torch.long
        ).to(self.device)
    
        att_mask = torch.tensor(
            attention_masks,
            dtype=torch.long
        ).to(self.device)
    
        # ============================================
        # Teacher Forcing Shift
        # ============================================

        input_tokens = tokens[:, :-1]    
        target_tokens = tokens[:, 1:]
        input_attention_mask = att_mask[:, :-1]

        return input_tokens, target_tokens, input_attention_mask
        

    def forward(self, input_ids, attention_mask):

        token_embeddings = self.embedding.forward(input_ids)

        x = self.PE.forward(token_embeddings)

        x = self.dropout(x)

        for block in self.decoder_blocks:
            x = block(x, attention_mask)

        x = self.final_layer_norm.forward(x)

        x = self.dropout(x)

        logits = self.output_projection(x)
        
        return logits


    def train_step(self, train_data, val_data):
    
        self.train()
    
        train_input_tokens, train_target_tokens, train_input_attention_mask = self.preprocess(train_data)
    
        # ============================================
        # Forward Pass
        # ============================================
    
        logits = self.forward(
            train_input_tokens,
            train_input_attention_mask
        )
    
        # ============================================
        # Loss
        # ============================================
    
        B, T, V = logits.shape
    
        train_loss = self.loss_fn(
            logits.view(B * T, V),
            train_target_tokens.reshape(B * T)
        )
    
        # ============================================
        # Backprop
        # ============================================
    
        self.optimizer.zero_grad()
        train_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.parameters(),
            1.0
        )
    
        self.optimizer.step()

        # ============================================
        # Validation
        # ============================================

        self.eval()
        
        with torch.no_grad():

            val_input_tokens, val_target_tokens, val_input_attention_mask = self.preprocess(val_data)
            
            val_logits = self.forward(
                val_input_tokens,
                val_input_attention_mask
            )
        
            B, T, V = val_logits.shape
        
            val_loss = self.loss_fn(
                val_logits.view(B * T, V),
                val_target_tokens.reshape(B * T)
            )
    
        return train_loss.item(), val_loss.item()


    def predict(self):
        pass

###### 

In [13]:
config = {
    "num_merges": 4000,
    "max_seq_len": 256,
    "embedding_dim": 256,
    "num_att_heads": 4,
    "num_decoder_blocks": 6,
    
    "epochs": 20,
    "warm_up_epochs": 3,
    
    "batch_size": 16,
    "learning_rate": 3e-4,
    "dropout_rate": 0.1,
    
    "patience": 5,
    "train_val_diff": 0.1,
    "val_improvement_threshold": 0.01,
    "skip_train_val_overfit_check": True,
    
    "gelu_const": math.sqrt(2.0 / math.pi),

    "train_vocab": True,
    "vocab_save_path": "../data/train_data_1.pkl",
    
    "train_dataset_path": "../data/wiki-2/train.txt",
    "test_dataset_path": "../data/wiki-2/test.txt",

    "model_save_path": "../models/mini_llm_checkpoint.pt"
}

In [ ]:
mini_llm = Mini_LLM(config)

Merge 1: (116, 104) -> 256
Merge 2: (105, 110) -> 257
Merge 3: (256, 101) -> 258
Merge 4: (97, 110) -> 259
Merge 5: (101, 114) -> 260


In [ ]:
mini_llm.to(mini_llm.device)

###### 

In [ ]:
class TextDataset(Dataset):

    def __init__(self, texts):

        self.texts = texts

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        return self.texts[idx]

In [ ]:
# Load train and val from txt files
with open(config["train_dataset_path"], "r") as f:
    train_data = f.read()

with open(config["test_dataset_path"], "r") as f:
    val_data = f.read()

train_texts = [t for t in train_data.split('\n') if t.strip()]
val_texts   = [t for t in val_data.split('\n') if t.strip()]

train_dataset = TextDataset(train_texts)
val_dataset   = TextDataset(val_texts)

# No random_split needed — already split
train_loader = DataLoader(
    train_dataset, 
    batch_size=config["batch_size"], 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,   
    batch_size=config["batch_size"], 
    shuffle=False
)

In [ ]:
def save_model(model, model_save_path):
    checkpoint = {
    
        "model_state_dict":
            model.state_dict(),
    
        "optimizer_state_dict":
            model.optimizer.state_dict(),
    
        "config":
            config,
    
        "epoch":
            epoch
    }
    
    torch.save(
        checkpoint,
        model_save_path
    )

    print(f"Model save at location: {model_save_path}")

In [ ]:
class EarlyStopping:
    
    def __init__(self, patience, train_val_diff, val_improvement_threshold, warm_up_epochs, skip_train_val_overfit_check):
        self.patience = patience
        self.train_val_diff = train_val_diff
        self.val_improvement_threshold = val_improvement_threshold
        self.warm_up_epochs = warm_up_epochs
        self.skip_train_val_overfit_check = skip_train_val_overfit_check
        self.counter = 0
        self.best_val_loss = float('inf')
        
    def check_performance(self, epoch, train_loss, val_loss):

        # Check overfitting: val loss diverging from train loss
        if not self.skip_train_val_overfit_check:
            overfitting = (val_loss - train_loss) > self.train_val_diff
        else:
            overfitting = False
        
        # Check plateau: val loss not improving enough
        val_improvement = self.best_val_loss - val_loss
        not_improving = val_improvement < self.val_improvement_threshold
        
        if overfitting or not_improving:
            if epoch > self.warm_up_epochs:
                self.counter += 1
                print(f"No improvement ({self.counter}/{self.patience})")
            else:
                print(f"No improvement (warm-up epoch - {epoch}/{self.warm_up_epochs})")
        else:
            self.best_val_loss = val_loss
            self.counter = 0  # reset if improvement found
            
        if self.counter >= self.patience:
            print(f"\nEarly stopping triggered after {self.patience} epochs without improvement.")
            return True, False

        if overfitting or not_improving:
            return False, False
            
        return False, True

In [ ]:
es = EarlyStopping(
    config["patience"], 
    config["train_val_diff"], 
    config["val_improvement_threshold"],
    config["warm_up_epochs"],
    config["skip_train_val_overfit_check"]
)

for epoch in range(config["epochs"]):

    total_train_loss = 0
    total_val_loss = 0

    train_loss_lst = []
    val_loss_lst = []

    for train_batch, val_batch in zip(train_loader, val_loader):

        train_loss, val_loss = mini_llm.train_step(
            train_batch,
            val_batch
        )

        total_train_loss += train_loss
        total_val_loss += val_loss

        train_loss_lst.append(train_loss)
        val_loss_lst.append(val_loss)

    avg_train_loss = (
        total_train_loss / len(train_loader)
    )

    avg_val_loss = (
        total_val_loss / len(val_loader)
    )

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

    stop_training, improvement = es.check_performance(epoch, avg_train_loss, avg_val_loss)
    if stop_training:
        break

    if improvement:
        save_model(mini_llm, config["model_save_path"])

    print("\n")